In [11]:
import numpy as np
from numba import cuda
import math
import time

print("CUDA Available:", cuda.is_available())

CUDA Available: True


In [12]:
N = 10_000_000

input_vector = np.random.rand(N).astype(np.float32)

print("Dataset Size:", len(input_vector))
print("Memory Usage:", input_vector.nbytes / (1024*1024), "MB")

Dataset Size: 10000000
Memory Usage: 38.14697265625 MB


In [13]:
@cuda.jit
def square_vector(input_arr, output_arr):
    idx = cuda.grid(1)

    if idx < input_arr.size:
        output_arr[idx] = input_arr[idx] * input_arr[idx]

In [14]:
start_cpu = time.time()

cpu_output = input_vector * input_vector

cpu_time = time.time() - start_cpu

print("CPU Time:", cpu_time, "seconds")

CPU Time: 0.014469146728515625 seconds


In [15]:
start_upload = time.time()

d_input = cuda.to_device(input_vector)
d_output = cuda.device_array_like(input_vector)

cuda.synchronize()

upload_time = time.time() - start_upload

print("GPU Upload Time:", upload_time, "seconds")

GPU Upload Time: 0.011035919189453125 seconds


In [16]:
square_vector[1, 1](d_input, d_output)
cuda.synchronize()

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [17]:
threads_per_block = 256
blocks_per_grid = math.ceil(N / threads_per_block)

start_kernel = time.time()

square_vector[blocks_per_grid, threads_per_block](
    d_input,
    d_output
)

cuda.synchronize()

kernel_time = time.time() - start_kernel

print("GPU Kernel Time:", kernel_time, "seconds")

GPU Kernel Time: 0.0018668174743652344 seconds


In [18]:
start_download = time.time()

gpu_output = d_output.copy_to_host()

download_time = time.time() - start_download

print("GPU Download Time:", download_time, "seconds")

GPU Download Time: 0.02191948890686035 seconds


In [19]:
correct = np.allclose(cpu_output, gpu_output)

print("Results Match:", correct)

Results Match: True


In [20]:
gpu_total_time = (
    upload_time +
    kernel_time +
    download_time
)

print("\n========== PERFORMANCE REPORT ==========")

print(f"CPU Time               : {cpu_time:.6f} sec")
print(f"GPU Upload Time        : {upload_time:.6f} sec")
print(f"GPU Kernel Time        : {kernel_time:.6f} sec")
print(f"GPU Download Time      : {download_time:.6f} sec")
print(f"GPU Total Time         : {gpu_total_time:.6f} sec")

print(f"\nKernel Speedup         : {cpu_time/kernel_time:.2f}x")
print(f"Overall GPU Speedup    : {cpu_time/gpu_total_time:.2f}x")


========== PERFORMANCE REPORT ==========
CPU Time               : 0.014469 sec
GPU Upload Time        : 0.011036 sec
GPU Kernel Time        : 0.001867 sec
GPU Download Time      : 0.021919 sec
GPU Total Time         : 0.034822 sec

Kernel Speedup         : 7.75x
Overall GPU Speedup    : 0.42x
